# MAGMA Gene/Pathway Analysis Processor & Visualizer

## Purpose
Processes MAGMA gene/pathway analysis results across multiple diseases. Applies False Discovery Rate (FDR) corrections (globally or localized to specific gene ontology terms like GOBP/GOCC/GOMF), extracts custom-selected pathways, and groups them programmatically by dominant disease intensity for matrix-based -log10(P-value) heatmap visualization.

## Input
* Raw MAGMA pathway output files matching `*_gene_analysis.gsa.out`.
* Alternatively, a pre-compiled input Excel workbook containing historical disease runs (e.g., `magma_results_05012026_inge.xlsx`).

## Process
1. **Parse and extract** text data fields from MAGMA log file directories.
2. **Adjust** downstream columns using the Benjamini-Hochberg (`fdr_bh`) correction.
3. **Filter** datasets by specific manual inclusions/exclusions of pathways.
4. **Matrix-transform** the long-form dataframes into 2D tables matching a predefined clinical disease order.
5. **Order pathways dynamically** by grouping them alongside their high-signal diseases.
6. **Construct** a custom truncated color gradient map with overlapping text annotations.

## Output
* Compiled backup workbook: `magma_results_16_04_2026.xlsx`.
* Publication-ready vector graphics file: `magma_heatmap.pdf`.

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.stats.multitest import multipletests

In [ ]:
# ------------------------------------------------------------------------------
# 1. INITIAL SETUP & PRIMARY DATA LOADING (PART 1)
# ------------------------------------------------------------------------------

directory = '/home/..'
dfs_dict = {}

# Walk through the folder and subfolders to find raw outputs
for root, dirs, files in os.walk(directory):
    for file in files:
        if file.endswith('_gene_analysis.gsa.out'):
            file_path = os.path.join(root, file)
            
            # UNCOMMENT TO MONITOR LOG LOADING:
            # print(f'Reading file: {file_path}')
            
            # Read space-delimited text data while filtering comments
            df = pd.read_csv(file_path, sep=r'\s+', comment='#')
            
            # Apply global FDR correction
            pvals = df["P"].values
            _, pvals_fdr, _, _ = multipletests(pvals, method='fdr_bh')
            
            df["P_fdr"] = pvals_fdr
            df = df.sort_values(by="P_fdr", ascending=True)            
            dfs_dict[file] = df

# Shorten and standardize dictionary keys 
dfs_dict_renamed = {key.split('_')[0]: value for key, value in dfs_dict.items()}

# UNCOMMENT TO INSPECT PARSED DICTIONARY KEYS & DATAFRAME SAMPLES:
# print(dfs_dict_renamed.keys())
# display(dfs_dict_renamed['PDproxy'][['FULL_NAME','P_fdr']].head(10))
# display(dfs_dict_renamed['PD'][['FULL_NAME','P_fdr']].head(10))

In [ ]:
# ------------------------------------------------------------------------------
# 2. OUTPUT GENERATION (PART 1 EXCEL BACKUP)
# ------------------------------------------------------------------------------

output_path = "magma_results_16_04_2026.xlsx"

with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df in dfs_dict_renamed.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

# UNCOMMENT TO CONFIRM INITIAL EXCEL WRITE:
# print(f"Saved Excel file to {output_path}")

In [ ]:
# ------------------------------------------------------------------------------
# 3. SECONDARY DATA PROCESSING (INGE RE-RUN COHORT)
# ------------------------------------------------------------------------------

input_path = "magma_results_05012026_inge.xlsx"
# input_path = "magma_results_16_04_2026.xlsx"

xlsx = pd.ExcelFile(input_path)

# Reload dictionary structures from intermediate files
dfs_dict_renamed = {
    sheet_name: pd.read_excel(xlsx, sheet_name=sheet_name)
    for sheet_name in xlsx.sheet_names
}

# UNCOMMENT TO CHECK RE-LOADED SHEET KEY DICTIONARIES:
# print(dfs_dict_renamed.keys())

In [ ]:
# ------------------------------------------------------------------------------
# 4. TARGETED FDR CORRECTION (GOBP / GOCC / GOMF)
# ------------------------------------------------------------------------------

for name, df in dfs_dict_renamed.items():
    # Clean column names defensively
    df.columns = df.columns.str.strip()
    df["P_fdr_GOBPGOCC"] = np.nan

    # Target specific Gene Ontology namespaces
    mask = df["FULL_NAME"].str.startswith(("GOBP", "GOCC", "GOMF"), na=False)

    if mask.any():
        pvals = df.loc[mask, "P"].values
        _, pvals_fdr, _, _ = multipletests(pvals, method="fdr_bh")
        df.loc[mask, "P_fdr_GOBPGOCC"] = pvals_fdr

# UNCOMMENT TO VALIDATE LOCAL GO COMPARTMENT ADJUSTMENTS:
# display(dfs_dict_renamed['PD'][['FULL_NAME','P','P_fdr','P_fdr_GOBPGOCC']].head(5))
# for name, df in dfs_dict_renamed.items():
#     print(name, df.columns.tolist())

In [ ]:
# ------------------------------------------------------------------------------
# 5. PATHWAY SUBSET FILTRATION
# ------------------------------------------------------------------------------

selected_genes = sorted(
    set(
        gene
        for df in dfs_dict_renamed.values()
        for gene in df.loc[df["Inclusion"] == True, "FULL_NAME"]
    )
)

# Remove predefined non-target terms
if 'GOBP_ADAPTIVE_IMMUNE_RESPONSE' in selected_genes:
    selected_genes.remove('GOBP_ADAPTIVE_IMMUNE_RESPONSE')
if 'GOBP_REGULATION_OF_SULFUR_METABOLIC_PROCESS' in selected_genes:
    selected_genes.remove('GOBP_REGULATION_OF_SULFUR_METABOLIC_PROCESS')    

# UNCOMMENT TO REVIEW FINAL CLEANED PATHWAY ARRAY SELECTIONS:
# print(selected_genes)

In [ ]:
# ------------------------------------------------------------------------------
# 6. MATRIX RE-SHAPING & SIGNAL SORTING
# ------------------------------------------------------------------------------

fdr = 'P_fdr_GOBPGOCC'
fdr = 'P_fdr'

# Restructure across target conditions
long_df = (
    pd.concat([
        df.assign(disease=disease)[["FULL_NAME", "P", fdr, "disease"]]
        for disease, df in dfs_dict_renamed.items()
    ])
    .query("FULL_NAME in @selected_genes")
)

# Color Metric Grid Creation: P-value representation
heatmap_p = (
    long_df
    .pivot_table(
        index="disease",
        columns="FULL_NAME",
        values="P",
        aggfunc="min"
    )
)
heatmap_p_log = -np.log10(heatmap_p)

# Significance Metric Grid Creation: FDR tracking
heatmap_sig = (
    long_df
    .pivot_table(
        index="disease",
        columns="FULL_NAME",
        values=fdr,
        aggfunc="min"
    )
)
sig_mask = heatmap_sig < 0.05

# Organize according to disease presentation parameters
disease_order = ["AD", "PDproxy", "DLB", "FTD", "ALS", 'NDEG', "MS", "MSA", "MDD", "BP", "SCZ"]

heatmap_p = heatmap_p.reindex(disease_order)
heatmap_p_log = heatmap_p_log.reindex(disease_order)
heatmap_sig = heatmap_sig.reindex(disease_order)
sig_mask = sig_mask.reindex(disease_order)

# Group terms dynamically on the axis using peak signal intensities
dominant_disease = heatmap_p_log.idxmax(axis=0)
dominant_disease = pd.Categorical(dominant_disease, categories=disease_order, ordered=True)

gene_order_df = pd.DataFrame({
    "gene": heatmap_p_log.columns,
    "dominant_disease": dominant_disease,
    "max_signal": heatmap_p_log.max(axis=0)
})

gene_order_df = gene_order_df.sort_values(by=["dominant_disease", "max_signal"], ascending=[True, False])
ordered_genes = gene_order_df["gene"].tolist()

# Apply the clustered layout across downstream calculation frames
heatmap_p_log = heatmap_p_log[ordered_genes]
sig_mask = sig_mask[ordered_genes]
heatmap_sig = heatmap_sig[ordered_genes]

# Build character overlay string matrices
annot = heatmap_sig.applymap(lambda x: "*" if x < 0.05 else ("o" if x < 0.1 else ""))

In [ ]:
# ------------------------------------------------------------------------------
# 7. COLORMAP CONFIGURATIONS & HEATMAP VISUALIZATION
# ------------------------------------------------------------------------------

def truncate_colormap(cmap, minval=0.5, maxval=1.0, n=256):
    return mcolors.LinearSegmentedColormap.from_list(
        "truncated_" + cmap.name,
        cmap(np.linspace(minval, maxval, n))
    )

cmap_trunc = truncate_colormap(plt.cm.Spectral, minval=0.5, maxval=1.0)

# Establish aesthetic colors and slice limits
grey = "#F0E6EB"
blue = "#2A418F"   
cmap_grey_to_blue = LinearSegmentedColormap.from_list("grey_to_blue", [grey, blue])

base_cmap = sns.color_palette("Spectral_r", as_cmap=True)
colors = base_cmap(np.linspace(0.5, 1, 256))
cmap_red_spectral = LinearSegmentedColormap.from_list("Spectral_r_redhalf", colors)

# Build Heatmap
vmax = 5
plt.figure(figsize=(12, 10))
ax = sns.heatmap(
    heatmap_p_log.T,
    cmap=cmap_red_spectral,
    linewidths=0.5,
    linecolor="grey",
    vmin=0,
    vmax=vmax,    
    cbar_kws={"label": "-log10(P-value)"},
    annot=annot.T,
    fmt="",
    annot_kws={"color": "white", "fontsize": 16}    
)

# Position labels cleanly
ax.xaxis.tick_top()
ax.xaxis.set_label_position("top")
plt.xticks(rotation=45, ha="left")
plt.xlabel("Disease")
plt.ylabel("Pathway")
plt.tight_layout()

# Configure vector configurations for Adobe Illustrator compatibility
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# Export and display visual arrays
plt.savefig("magma_heatmap.pdf", format="pdf", bbox_inches="tight")
plt.show()